Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Chequeo del ejercicio a mano**

In [1]:
import torch
import torch.nn as nn

### **Autograd**

PyTorch nos permite calcular gradientes automáticamente. Veamos cómo.

In [2]:
a = torch.tensor(5.0, requires_grad = True)
b = torch.tensor(2.0, requires_grad = True)
c = torch.tensor(10.0, requires_grad = True)

In [3]:
d = (a + b)
e = d * c

Ejecutamos *backpropagation* con la llamada al método `backward`. Recordemos que backpropagation sirve no sólo para redes neuronales, sino para calcular el gradiente de una función arbitraria.

In [4]:
e.backward()

In [5]:
print('Gradiente de e con respecto a a:', a.grad)
print('Gradiente de e con respecto a b:', b.grad)
print('Gradiente de e con respecto a c:', c.grad)

Gradiente de e con respecto a a: tensor(10.)
Gradiente de e con respecto a b: tensor(10.)
Gradiente de e con respecto a c: tensor(7.)


Esos gradientes nos dicen cuánto cambia `e` ante un cambio en `a`, `b` o `c`, respectivamente.

### **El ejercicio a mano**

Veamos cómo reproducir en PyTorch la pasada *forward* y la pasada *backward* que vimos antes de forma manual.

##### **Primeros pasos**

Primero, definimos la red, con los pesos iniciales:

In [6]:
class Net(nn.Module):
    
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(1, 1)
        self.fc2 = nn.Linear(1, 1)
    
    def weights_initialization(self):
        self.fc1.weight = nn.Parameter(torch.tensor([[0.15]]))
        self.fc1.bias = nn.Parameter(torch.tensor([0.20]))
        
        self.fc2.weight = nn.Parameter(torch.tensor([[0.1]]))
        self.fc2.bias = nn.Parameter(torch.tensor([0.5]))
        
        # Cabe aclarar que, normalmente, los pesos no se definen a mano, sino
        # que se inicializan de forma aleatoria.
    
    def forward(self, x):
        x = nn.Sigmoid()(self.fc1(x))
        x = nn.Sigmoid()(self.fc2(x))
        return x

Después, establecemos la tasa de aprendizaje como 0,5:

In [7]:
learning_rate = 0.5

##### **Pasada forward**

In [8]:
net = Net()
print(net)
net.weights_initialization()

y_pred = net(torch.tensor([10.0]))

Net(
  (fc1): Linear(in_features=1, out_features=1, bias=True)
  (fc2): Linear(in_features=1, out_features=1, bias=True)
)


In [9]:
print(f'Predicción: {y_pred.item()}')

Predicción: 0.6421144604682922


Creamos la función de pérdida (MSE):

In [10]:
loss_fn = torch.nn.MSELoss(reduction = 'sum')

Definimos la salida esperada como 0,2:

In [11]:
y = torch.tensor([0.2])

In [12]:
loss = loss_fn(y_pred, y)
print(f'Loss: {loss.item()}')

Loss: 0.19546520709991455


##### **Pasada backward**

In [13]:
net.zero_grad()
loss.backward()

Observamos los gradientes después de la pasada backward:

In [14]:
with torch.no_grad():
    for param in net.parameters():
        if param.requires_grad:
            print(param.grad)
            param -= learning_rate * param.grad

tensor([[0.0265]])
tensor([0.0027])
tensor([[0.1718]])
tensor([0.2032])


Observamos los pesos después de la pasada backward:

In [15]:
print(f'w1: {net.fc1.weight.item()}')
print(f'b1: {net.fc1.bias.item()}')
print(f'w2: {net.fc2.weight.item()}')
print(f'b2: {net.fc2.bias.item()}')

w1: 0.13673053681850433
b1: 0.1986730545759201
w2: 0.014094136655330658
b2: 0.39840054512023926


In [16]:
y_pred2 = net(torch.tensor([10.0]))
print(f'Predicción después de la corrección: {y_pred2.item()}')

Predicción después de la corrección: 0.6011021137237549


In [17]:
loss = loss_fn(y_pred2, y)
print(f'Loss después de la corrección: {loss}')

Loss después de la corrección: 0.16088292002677917


Vemos que la *loss* bajó y, efectivamente, la segunda predicción está más cerca que la primera del valor esperado.

A su vez, los resultados coinciden con los que habíamos obtenido antes resolviendo el ejercicio a mano.